# Single-Cell RNA-seq of the Breast-Tumour Microenvironment (Scanpy)

**Question.** What cell types make up a human breast tumour, and can we recover them, unsupervised, from single-cell transcriptomes?

**Data.** Real human breast-cancer cells fetched **live via the CELLxGENE Census API** (CZI; TileDB-SOMA over S3) — not a packaged tutorial dataset. Query: `tissue_general == 'breast' and disease != 'normal' and is_primary_data == True`; ~5,000 cells sampled (Census version pinned + fixed random seed → reproducible). The Census also ships **expert `cell_type` labels**, which we ignore during clustering and use only to *validate* our own marker-based annotation.

**Pipeline.** QC → normalize/log1p → highly-variable genes → scale → PCA → kNN graph → UMAP → Leiden clustering → marker genes (Wilcoxon) → cell-type annotation → cross-tab against the expert labels.

Companion R/Seurat twin: `single_cell_tme.R` (reads the same cells, exported below).

## 0. Fetch real breast-tumour cells via the Census API
Returns raw UMI counts as an **AnnData** (rows = cells, cols = genes); `feature_name` gives HGNC symbols so `MT-` detection and canonical markers work. First run downloads from S3 (the scattered-cell sample can take a few minutes); pinning the version + seed makes the exact 5,000 cells reproducible.

In [ ]:
%pip install cellxgene-census scanpy leidenalg -q
import scanpy as sc, cellxgene_census, pandas as pd, numpy as np, os
os.makedirs("results_py", exist_ok=True); os.makedirs("data", exist_ok=True)
sc.settings.figdir = "results_py"; sc.settings.verbosity = 1

FILT = "tissue_general == 'breast' and disease != 'normal' and is_primary_data == True"
census = cellxgene_census.open_soma(census_version="2025-11-08")            # pinned for reproducibility
obs = cellxgene_census.get_obs(census, "Homo sapiens", value_filter=FILT,
                               column_names=["soma_joinid", "cell_type", "disease"])
sel = obs["soma_joinid"].sample(n=min(5000, len(obs)), random_state=0).to_numpy()
adata = cellxgene_census.get_anndata(
            census, organism="Homo sapiens", obs_coords=sel,
            obs_column_names=["cell_type", "disease", "tissue"],
            var_column_names=["feature_id", "feature_name"])
census.close()
adata.var_names = adata.var["feature_name"].astype(str); adata.var_names_make_unique()
print(adata)
print(adata.obs["cell_type"].value_counts().head(15))

### Export the same cells in 10x format (for the R/Seurat twin)
Writes `data/tumor_10x/` (matrix + features + barcodes + Census labels) **from raw counts**, before any processing changes `adata.X`. The R twin reads this, so both languages analyse the identical cells without needing the (build-fragile) R Census package.

In [ ]:
import scipy.io as sio, gzip, shutil
d = "data/tumor_10x"; os.makedirs(d, exist_ok=True)
sio.mmwrite(f"{d}/matrix.mtx", adata.X.T.astype("int32"))                    # 10x wants genes x cells
pd.DataFrame({"id": adata.var["feature_id"].values,
              "name": adata.var["feature_name"].values,
              "type": "Gene Expression"}).to_csv(f"{d}/features.tsv", sep="\t", header=False, index=False)
pd.Series(adata.obs_names).to_csv(f"{d}/barcodes.tsv", sep="\t", header=False, index=False)
adata.obs[["cell_type"]].to_csv(f"{d}/cell_meta.csv")                        # expert labels for the R crosstab
for f in ["matrix.mtx", "features.tsv", "barcodes.tsv"]:
    with open(f"{d}/{f}", "rb") as fi, gzip.open(f"{d}/{f}.gz", "wb") as fo: shutil.copyfileobj(fi, fo)
    os.remove(f"{d}/{f}")
print("wrote", d)

## 1. Quality control
Flag mitochondrial genes, compute per-cell metrics, and drop technical junk: empty droplets (too few genes), rarely-detected genes, and dying cells (high mito%). Thresholds are looser than for PBMCs (genes < 6000, mito < 15%) because tumour dissociation is harsher and malignant cells are large/transcriptionally active — a strict cut would delete real tumour cells.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[(adata.obs["n_genes_by_counts"] < 6000) & (adata.obs["pct_counts_mt"] < 15)].copy()
print(adata.shape, "cells x genes after QC")

## 2. Normalize + log-transform
Library-size normalize each cell to 10,000 counts (so cells sequenced to different depths are comparable), then `log1p` to stabilise variance. Keep the log-normalized full matrix in `adata.raw` for marker plots.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata

## 3. Highly variable genes
Keep the most variable genes — the cell-to-cell signal that separates cell types — and drop uninformative near-constant genes.

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata = adata[:, adata.var["highly_variable"]].copy()
print(adata.shape, "cells x HVGs")

## 4. Scale + PCA
Z-score each gene (clip at 10) so none dominates by magnitude, then compress to principal components capturing the main axes of variation.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver="arpack")

## 5. kNN graph + UMAP
Build the nearest-neighbour graph in PC space (used by both clustering and UMAP), then embed to 2-D for visualization. UMAP preserves local neighbourhoods; inter-cluster distances are **not** quantitative.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

## 6. Leiden clustering
Community detection on the kNN graph (maximise modularity). `resolution` sets granularity; clusters are validated by coherent markers below, not by their count.

In [ ]:
sc.tl.leiden(adata, resolution=0.5)
sc.pl.umap(adata, color=["leiden"], save="_leiden.png")
print(adata.obs["leiden"].value_counts())

## 7. Marker genes per cluster
Wilcoxon rank-sum test of each cluster vs the rest → genes over-expressed in that cluster identify it.

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=25, sharey=False, save="_markers.png")
top = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(10)
top.to_csv("results_py/markers_top10.csv", index=False)
for g in adata.uns["rank_genes_groups"]["names"].dtype.names:
    print(g, list(adata.uns["rank_genes_groups"]["names"][g][:6]))

## 8. Annotate clusters → cell types, and validate against expert labels
Each cluster is labelled from its own top markers using canonical breast-TME genes:
KRT8/18/19, EPCAM, GATA3 → malignant/epithelial; COL1A1/DCN/LUM, ACTA2/TAGLN → fibroblast (CAF); VWF/PECAM1/MMRN2 → endothelial; CD3D/CD3E → T cell; MS4A1/CD79A → B cell; IGKC/IGHG/MZB1 → plasma; CD68/LYZ/TYROBP → macrophage; TPSAB1/CPA3 → mast. Several clusters share a coarse label (expected at 24 clusters). The final `crosstab` against the Census `cell_type` is the validation: a near-block-diagonal table means our unsupervised calls agree with the expert annotation.

In [ ]:
mapping = {
    "0":  "T cell",           # B2M/HLA-high + CD3D
    "1":  "malignant cell",   # KRT19/KRT18/KRT8/GATA3
    "2":  "macrophage",       # CD68/LYZ/TYROBP/FCER1G
    "3":  "malignant cell",   # KRT19 + ribosomal-high
    "4":  "fibroblast",       # COL1A1/COL1A2/DCN/LUM
    "5":  "T cell",           # CD3E/IL32/CORO1A
    "6":  "endothelial cell", # VWF/IGFBP7/RAMP2
    "7":  "malignant cell",   # KRT14/KRT7/FXYD3
    "8":  "IgG plasma cell",  # IGKC/IGHG1/MZB1/JCHAIN
    "9":  "malignant cell",   # TACSTD2/KRT6B/KRT14/KRT17
    "10": "fibroblast",       # ACTA2/TAGLN/MYL9 (myofibroblast/CAF)
    "11": "B cell",           # MS4A1/CD79A/CD37
    "12": "T cell",           # HSP/JUND stress state (immune-leaning)
    "13": "malignant cell",   # TRPS1/PBX1/DACH1 luminal TFs
    "14": "malignant cell",   # GATA3/BCAM/ERBB3 luminal
    "15": "malignant cell",   # CRABP2 + ribo
    "16": "malignant cell",   # CEACAM7/TKTL1 proliferating
    "17": "malignant cell",   # epithelial-leaning
    "18": "malignant cell",   # epithelial-leaning
    "19": "mast cell",        # TPSAB1/TPSB2/CPA3/MS4A2
    "20": "endothelial cell", # LDB2/PTPRM
    "21": "endothelial cell", # MMRN2 (lymphatic EC)
    "22": "malignant cell",   # lincRNA-heavy / low-signal cluster
    "23": "T cell",           # SKAP1/ARHGAP15/AOAH
}
adata.obs["my_label"] = adata.obs["leiden"].map(mapping).astype("category")
sc.pl.umap(adata, color=["my_label", "cell_type"], save="_celltypes.png")
adata.obs["my_label"].value_counts().to_csv("results_py/celltype_counts.csv")
ct = pd.crosstab(adata.obs["my_label"], adata.obs["cell_type"])
ct.to_csv("results_py/crosstab_vs_census.csv")
print(ct)

## Interpretation

Starting from a raw cell × gene count matrix of real breast-tumour cells, an unsupervised pipeline (QC → normalize → HVGs → PCA → kNN → Leiden) recovered the expected compartments of a solid tumour: **malignant/epithelial cells** (keratins, GATA3, luminal TFs), **fibroblasts/CAFs** (collagens, ACTA2/TAGLN), **endothelial cells** (VWF/PECAM1, lymphatic MMRN2), **T cells** (CD3D/CD3E), **B and plasma cells** (MS4A1/CD79A; IGKC/MZB1), **macrophages** (CD68/LYZ), and **mast cells** (TPSAB1/CPA3). Cross-tabulating these marker-based labels against the Census expert `cell_type` gives a near-block-diagonal table — the unsupervised calls agree with independent expert annotation. Biologically this confirms that a tumour is an **ecosystem**: malignant epithelium embedded in stroma and immune cells, resolvable only at single-cell resolution (a bulk average would blend them and hide the immune contexture that drives prognosis and immunotherapy response).

**Caveats.** Results depend on QC thresholds, #HVGs, #PCs and Leiden resolution; UMAP geometry is not quantitative; annotation is marker-based (a supervised label on unsupervised clusters); the Census pool spans multiple donors/datasets, so a rigorous study would batch-integrate (Harmony/scVI) first; and calling an epithelial cluster *malignant* (vs normal epithelium) needs copy-number inference (inferCNV/CopyKAT), not markers alone. NK cells fold into the T-cell clusters at this resolution.

### Abstract
*Fetched ~5,000 real human breast-tumour single cells via the CELLxGENE Census API and ran a standard Scanpy pipeline (QC, normalization, HVG selection, PCA, kNN graph, UMAP, Leiden clustering, Wilcoxon marker detection). Marker-based annotation recovered the malignant, stromal (fibroblast, endothelial) and immune (T, B, plasma, macrophage, mast) compartments of the tumour microenvironment, and cross-validation against the Census expert `cell_type` labels showed strong agreement. Reproduced on the identical cells in R/Seurat.*